# urllib 基础爬虫教程

本教程介绍如何使用 Python 的 urllib 库进行基础网页爬取。

## 目录
1. 基础连接
2. 添加请求头
3. 使用代理
4. 正则表达式提取数据

## 1. 基础网页连接

最简单的网页请求示例

In [ ]:
import urllib.request
import time

def link_website(url):
    """连接网站并获取内容"""
    try:
        response = urllib.request.urlopen(url, timeout=3)
        result = response.read().decode('utf-8')
        
        print(f"URL: {response.geturl()}")
        print(f"状态码: {response.getcode()}")
        print(f"响应头信息:\n{response.info()}")
        
        return result
    except Exception as e:
        print(f"连接失败: {e}")
        return None

# 示例：连接百度
url = 'http://www.baidu.com'
content = link_website(url)

if content:
    print(f"\n获取到的内容长度: {len(content)} 字符")
    print(f"前200个字符:\n{content[:200]}")

## 2. 添加请求头（User-Agent）

模拟浏览器访问，避免被网站识别为爬虫

In [ ]:
import urllib.request

def fetch_with_headers(url):
    """使用自定义请求头访问网站"""
    # 常用的 User-Agent 列表
    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'
    ]
    
    headers = {
        'User-Agent': user_agents[0],
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8'
    }
    
    request = urllib.request.Request(url, headers=headers)
    
    try:
        response = urllib.request.urlopen(request, timeout=5)
        content = response.read().decode('utf-8')
        print(f"成功获取 {url}")
        return content
    except Exception as e:
        print(f"请求失败: {e}")
        return None

# 测试
url = 'http://httpbin.org/user-agent'
result = fetch_with_headers(url)
if result:
    print(result)

## 3. 使用代理

通过代理服务器访问网站

In [ ]:
import urllib.request

def fetch_with_proxy(url, proxy_addr):
    """使用代理访问网站
    
    Args:
        url: 目标网址
        proxy_addr: 代理地址，格式如 '127.0.0.1:8080'
    """
    proxy_handler = urllib.request.ProxyHandler({
        'http': f'http://{proxy_addr}',
        'https': f'https://{proxy_addr}'
    })
    
    opener = urllib.request.build_opener(proxy_handler)
    urllib.request.install_opener(opener)
    
    try:
        response = urllib.request.urlopen(url, timeout=10)
        content = response.read().decode('utf-8')
        print(f"通过代理成功访问: {url}")
        return content
    except Exception as e:
        print(f"代理访问失败: {e}")
        return None

# 示例（需要替换为实际可用的代理）
# proxy = '127.0.0.1:8080'
# result = fetch_with_proxy('http://httpbin.org/ip', proxy)
print("提示：使用代理需要提供实际可用的代理服务器地址")

## 4. 使用正则表达式提取数据

从网页中提取特定信息

In [ ]:
import re
import urllib.request
import time

class SimpleMovieCrawler:
    """简单的电影信息爬虫示例"""
    
    def __init__(self, url):
        self.url = url
        self.timeout = 5
    
    def fetch_content(self):
        """获取网页内容"""
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        request = urllib.request.Request(self.url, headers=headers)
        
        try:
            response = urllib.request.urlopen(request, timeout=self.timeout)
            return response.read().decode('utf-8')
        except Exception as e:
            print(f"获取内容失败: {e}")
            return None
    
    def extract_info(self, html_content, pattern):
        """使用正则表达式提取信息"""
        compiled_pattern = re.compile(pattern)
        results = compiled_pattern.findall(html_content)
        return results

# 示例：提取标题标签
print("正则表达式提取示例")
print("="*50)

# 模拟 HTML 内容
sample_html = """
<html>
    <div class="movie">
        <span class="title">流浪地球</span>
        <span class="score">8.5</span>
    </div>
    <div class="movie">
        <span class="title">三体</span>
        <span class="score">9.0</span>
    </div>
</html>
"""

# 提取电影标题
title_pattern = r'<span class="title">(.*?)</span>'
titles = re.findall(title_pattern, sample_html)
print(f"提取到的标题: {titles}")

# 提取评分
score_pattern = r'<span class="score">(.*?)</span>'
scores = re.findall(score_pattern, sample_html)
print(f"提取到的评分: {scores}")

## 5. 完整爬虫示例

综合运用以上技术

In [ ]:
import urllib.request
import re
import time
from datetime import datetime

class WebCrawler:
    """通用网页爬虫类"""
    
    def __init__(self, base_url, timeout=5):
        self.base_url = base_url
        self.timeout = timeout
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
    
    def fetch(self, url=None):
        """获取网页内容"""
        target_url = url or self.base_url
        request = urllib.request.Request(target_url, headers=self.headers)
        
        try:
            response = urllib.request.urlopen(request, timeout=self.timeout)
            content = response.read().decode('utf-8')
            print(f"✓ 成功获取: {target_url}")
            return content
        except Exception as e:
            print(f"✗ 获取失败: {e}")
            return None
    
    def extract(self, content, pattern):
        """提取数据"""
        if not content:
            return []
        return re.findall(pattern, content)
    
    def save(self, data, filename):
        """保存数据到文件"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"爬取时间: {timestamp}\n")
            f.write("="*50 + "\n")
            for item in data:
                f.write(f"{item}\n")
        
        print(f"✓ 数据已保存到: {filename}")

# 使用示例
print("爬虫类已定义，可以根据需要实例化使用")
print("\n示例代码:")
print("crawler = WebCrawler('http://example.com')")
print("content = crawler.fetch()")
print("data = crawler.extract(content, r'<title>(.*?)</title>')")
print("crawler.save(data, 'output.txt')")

## 练习题

1. 修改 User-Agent，尝试模拟不同的浏览器
2. 编写正则表达式提取网页中的所有链接
3. 添加重试机制，当请求失败时自动重试
4. 实现一个简单的爬虫，爬取指定网站的标题和描述

## 注意事项

- 遵守网站的 robots.txt 规则
- 控制爬取频率，避免对服务器造成压力
- 尊重网站的版权和用户隐私
- 某些网站可能需要登录或有反爬虫机制